# Gemma 2B IT LoRA Training

This Colab-style notebook trains a Gemma 2B IT LoRA adapter, uploads the adapter to Hugging Face, and optionally registers the model in Burstchester.

In [ ]:
#@title 1. Training settings
REPO_URL = "https://github.com/tomongoose/burstchester.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
REPO_DIR = "/content/burstchester" #@param {type:"string"}
DATASET_IDS = "a2492509-d5ea-4234-9197-cd7fdebbf801,legal-ko" #@param {type:"string"}
BASE_MODEL = "google/gemma-2b-it" #@param {type:"string"}
TRAIN_COMMAND = "train-gemma-2b-it-lora" #@param ["train-gemma-2b-it-lora"]
TRAINING_METHOD = "lora" #@param ["lora"]
WORKSPACE = "/content/burstchester-training/gemma-2b-it-lora" #@param {type:"string"}
EPOCHS = "1" #@param {type:"string"}
BATCH_SIZE = "1" #@param {type:"string"}
MAX_SEQ_LENGTH = "128" #@param {type:"string"}
LORA_RANK = "8" #@param {type:"string"}
LORA_ALPHA = "16" #@param {type:"string"}
LORA_DROPOUT = "0.05" #@param {type:"string"}
OUTPUT_MODEL_REPO = "hf-user/gemma-2b-it-lora" #@param {type:"string"}
MODEL_TITLE = "Gemma 2B IT LoRA Legal Ko" #@param {type:"string"}
MODEL_POINT_COST = "30" #@param {type:"string"}
SKIP_REGISTER = False #@param {type:"boolean"}
SKIP_HF_UPLOAD = False #@param {type:"boolean"}

print("Settings loaded. Secrets are requested in the next cell.")

In [ ]:
# 2. Load secrets without saving them in notebook output.
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def load_secret(name, prompt, required=True):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(prompt)
    if required and not str(value or "").strip():
        raise ValueError(f"{name} is required.")
    if value:
        os.environ[name] = str(value).strip()
    return os.environ.get(name, "")

load_secret("BURSTCHESTER_ACCESS_TOKEN", "Burstchester access token: ")
load_secret("HF_TOKEN", "Hugging Face token: ")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("Secrets configured in environment.")

In [ ]:
# 3. Clone or update the repository.
from pathlib import Path
import subprocess

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

subprocess.run(["git", "checkout", REPO_BRANCH], cwd=repo_dir, check=True)
%cd {REPO_DIR}

In [ ]:
# 4. Install CLI and model dependencies in this Colab runtime.
import subprocess

# Colab can include an old optional torchao package. Recent peft detects it
# and fails unless torchao is >0.16, so remove it instead of upgrading torch.
subprocess.run(["python", "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

subprocess.run([
    "python", "-m", "pip", "install", "-q", "-U",
    "transformers",
    "accelerate",
    "datasets",
    "peft",
    "huggingface_hub",
], check=True)
subprocess.run([
    "python", "-m", "pip", "install", "-q", "-U", "--no-deps",
    "bitsandbytes",
], check=True)
print("CLI/model dependencies installed.")


In [ ]:
# 5. Download and run a short base-model smoke test.
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. In Colab, open Runtime > Change runtime type and select a GPU.")

model_name = BASE_MODEL
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=os.environ.get("HF_TOKEN") or None,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=os.environ.get("HF_TOKEN") or None,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

prompt = "Write one short sentence about dataset quality."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=32, do_sample=False)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Base model downloaded and smoke-tested:", model_name)


In [ ]:
# 6. Validate dataset settings and export training environment.
import os
from huggingface_hub import HfApi

if not DATASET_IDS.strip():
    raise ValueError("Set DATASET_IDS to your Burstchester dataset ID(s) before training.")
if not OUTPUT_MODEL_REPO.strip() and not SKIP_REGISTER:
    raise ValueError("Set OUTPUT_MODEL_REPO or enable SKIP_REGISTER.")
if not MODEL_TITLE.strip() and not SKIP_REGISTER:
    raise ValueError("Set MODEL_TITLE before registering the model in Burstchester.")
if OUTPUT_MODEL_REPO.startswith("hf-user/") and not SKIP_REGISTER:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN is required to replace the hf-user placeholder.")
    hf_user = HfApi(token=token).whoami().get("name")
    if not hf_user:
        raise ValueError("Could not resolve Hugging Face username from HF_TOKEN.")
    OUTPUT_MODEL_REPO = f"{hf_user}/{OUTPUT_MODEL_REPO.split('/', 1)[1]}"
    print("Resolved OUTPUT_MODEL_REPO:", OUTPUT_MODEL_REPO)

os.environ["DATASET_IDS"] = DATASET_IDS
os.environ["BASE_MODEL"] = BASE_MODEL
os.environ["TRAIN_COMMAND"] = TRAIN_COMMAND
os.environ["TRAINING_METHOD"] = TRAINING_METHOD
os.environ["WORKSPACE"] = WORKSPACE
os.environ["EPOCHS"] = EPOCHS
os.environ["BATCH_SIZE"] = BATCH_SIZE
os.environ["MAX_SEQ_LENGTH"] = MAX_SEQ_LENGTH
os.environ["LORA_RANK"] = LORA_RANK
os.environ["LORA_ALPHA"] = LORA_ALPHA
os.environ["LORA_DROPOUT"] = LORA_DROPOUT
os.environ["OUTPUT_MODEL_REPO"] = OUTPUT_MODEL_REPO
os.environ["MODEL_TITLE"] = MODEL_TITLE
os.environ["MODEL_POINT_COST"] = MODEL_POINT_COST
os.environ["SKIP_REGISTER"] = "1" if SKIP_REGISTER else "0"
os.environ["SKIP_HF_UPLOAD"] = "1" if SKIP_HF_UPLOAD else "0"

print("Training dataset IDs:", os.environ["DATASET_IDS"])
print("Output model title:", os.environ["MODEL_TITLE"])
print("Output model repo:", os.environ["OUTPUT_MODEL_REPO"])


In [ ]:
# 7. Run training, upload, and registration.
!bash cli/scripts/colab-train-and-register.sh
